In [ ]:
# Step 1: Load the CSV
In this step, we will import the necessary data analysis library, `pandas`, and load our dataset into a DataFrame. Then we will display the first few rows using `.head()` to ensure it loaded correctly.
</VSCode.Cell>

<VSCode.Cell language="python">
import pandas as pd

# Load the dataset assuming it is in the same directory
df = pd.read_csv('results.csv')
``
# Display the first few rows
df.head()
</VSCode.Cell>

<VSCode.Cell language="markdown">
# Basic Exploration
Here we will explore the fundamental properties of the dataset.

- **Match Count**: We determine the number of rows in the DataFrame to find the total matches.
- **Earliest and Latest Year**: By converting the `date` column into a `datetime` object, we can extract the `year` component and find the minimum (earliest) and maximum (latest) years.
- **Unique Countries**: We'll use `.nunique()` on the `country` column to see how many distinct hosting countries exist in the records.
- **Most Frequent Home Team**: By finding the mode (or using `.value_counts().index[0]`) on the `home_team` column, we can see which team has played the most fixtures at home.
</VSCode.Cell>

<VSCode.Cell language="python">
# Total matches
total_matches = len(df)
print(f"Total matches in dataset: {total_matches}")

# Earliest and Latest Year
df['date'] = pd.to_datetime(df['date'])
earliest_year = df['date'].dt.year.min()
latest_year = df['date'].dt.year.max()
print(f"Earliest year: {earliest_year}")
print(f"Latest year: {latest_year}")

# Unique countries
unique_countries = df['country'].nunique()
print(f"Unique countries: {unique_countries}")

# Most frequent home team
most_frequent_home = df['home_team'].value_counts().index[0]
print(f"Most frequent home team: {most_frequent_home}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
# Goals Analysis
Next, we focus on goal-scoring patterns.

- **Total Goals**: We create a new column `total_goals` by adding `home_score` and `away_score`.
- **Average Goals**: We take the mean of this new column to get the average goals per match.
- **Highest Scoring Match**: We use the maximum value of `total_goals` to filter the DataFrame and extract the row details of the highest scoring fixture.
- **Home vs Away Breakdown**: We sum all `home_score` and `away_score` columns respectively, comparing them to see if home or away teams score more.
- **Mode of Total Goals**: Using `.mode()`, we check the most common number of total goals scored in a match.
</VSCode.Cell>

<VSCode.Cell language="python">
# Create total_goals column
df['total_goals'] = df['home_score'] + df['away_score']

# Average goals per match
average_goals = df['total_goals'].mean()
print(f"Average goals per match: {average_goals:.2f}")

# Highest scoring match details
highest_scoring_match = df.loc[df['total_goals'].idxmax()]
print("\nHighest Scoring Match Details:")
print(highest_scoring_match)

# Total home vs away goals
total_home_goals = df['home_score'].sum()
total_away_goals = df['away_score'].sum()

print(f"\nTotal Home Goals: {total_home_goals}")
print(f"Total Away Goals: {total_away_goals}")
if total_home_goals > total_away_goals:
    print("More goals are scored by the Home team.")
elif total_away_goals > total_home_goals:
    print("More goals are scored by the Away team.")
else:
    print("Home and Away goals are exactly equal.")

# Mode of total goals
mode_goals = df['total_goals'].mode()[0]
print(f"\nMost common total goals (mode): {mode_goals}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
# Match Results
Now we categorize the match outcomes into "Home Win", "Away Win", or "Draw".

- **Result Function**: We define `match_result(row)` and apply it across the DataFrame (setting `axis=1`) to yield a new `result` column.
- **Home Win Percentage**: We count the occurrences of "Home Win" and divide by the total matches to yield a percentage.
- **Home Advantage**: By comparing the count of Home Wins vs Away Wins, we can state whether playing at home gives a historical advantage.
- **Most Wins Historically**: We aggregate wins. A team gets a win when they are the home team and the result is a "Home Win", OR when they are the away team and the result is an "Away Win". We'll combine this data to find the overall most successful team.
</VSCode.Cell>

<VSCode.Cell language="python">
# Create match_result function
def match_result(row):
    if row['home_score'] > row['away_score']:
        return "Home Win"
    elif row['home_score'] < row['away_score']:
        return "Away Win"
    else:
        return "Draw"

# Apply function to create 'result' column
df['result'] = df.apply(match_result, axis=1)

# Percentage of home wins
result_counts = df['result'].value_counts()
home_wins_percent = (result_counts.get("Home Win", 0) / len(df)) * 100
print(f"Percentage of home wins: {home_wins_percent:.2f}%")

# Home advantage comparison
home_wins = result_counts.get("Home Win", 0)
away_wins = result_counts.get("Away Win", 0)
print(f"\nHome wins: {home_wins}, Away wins: {away_wins}")
if home_wins > away_wins:
    print("Statement: A significant home advantage appears to exist historically.")
else:
    print("Statement: There is little to no home advantage historically.")

# Determine which team has the most historical wins
# Filter to only matches with a winner
home_winners = df[df['result'] == 'Home Win']['home_team']
away_winners = df[df['result'] == 'Away Win']['away_team']

# Combine all winning teams
all_winners = pd.concat([home_winners, away_winners])
most_wins_team = all_winners.value_counts().index[0]
most_wins_count = all_winners.value_counts().iloc[0]

print(f"\nTeam with the most historical wins: {most_wins_team} ({most_wins_count} wins)")
</VSCode.Cell>

<VSCode.Cell language="markdown">
# Visualization
Finally, we build visualizations to better understand our dataset using `matplotlib.pyplot`.

- **Histogram of Total Goals**: Shows the distribution of goals scored per match. Setting `bins=15` clumps the data appropriately.
- **Bar Chart of Outcomes**: Displays the occurrences of the three match results.
- **Bar Chart of Top 10 Teams**: We take the top 10 from our aggregated wins list earlier, plot them, and rotate the x-axis labels (`plt.xticks(rotation=45)`) so the team names are fully readable.
</VSCode.Cell>

<VSCode.Cell language="python">
import matplotlib.pyplot as plt

# 1. Histogram of Total Goals
plt.figure(figsize=(10, 5))
plt.hist(df['total_goals'], bins=15, color='skyblue', edgecolor='black')
plt.title('Distribution of Total Goals Per Match')
plt.xlabel('Total Goals')
plt.ylabel('Frequency')
plt.show()

# 2. Bar chart of match outcomes
plt.figure(figsize=(8, 5))
result_counts.plot(kind='bar', color=['#4C72B0', '#55A868', '#C44E52'])
plt.title('Frequency of Match Outcomes')
plt.xlabel('Result')
plt.ylabel('Number of Matches')
plt.xticks(rotation=0)
plt.show()

# 3. Bar chart of the Top 10 teams by total wins
top_10_winners = all_winners.value_counts().head(10)

plt.figure(figsize=(12, 6))
top_10_winners.plot(kind='bar', color='coral')
plt.title('Top 10 Teams by Historical Wins')
plt.xlabel('Team / Country')
plt.ylabel('Total Wins')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()  # Ensure labels do not get cut off
plt.show()


In [ ]:
# Step 1: Load the CSV
In this step, we will import the necessary data analysis library, `pandas`, and load our dataset into a DataFrame. Then we will display the first few rows using `.head()` to ensure it loaded correctly.
</VSCode.Cell>

<VSCode.Cell language="python">
import pandas as pd

# Load the dataset assuming it is in the same directory
df = pd.read_csv('results.csv')

# Display the first few rows
df.head()
</VSCode.Cell>

<VSCode.Cell language="markdown">
# Basic Exploration
Here we will explore the fundamental properties of the dataset.

- **Match Count**: We determine the number of rows in the DataFrame to find the total matches.
- **Earliest and Latest Year**: By converting the `date` column into a `datetime` object, we can extract the `year` component and find the minimum (earliest) and maximum (latest) years.
- **Unique Countries**: We'll use `.nunique()` on the `country` column to see how many distinct hosting countries exist in the records.
- **Most Frequent Home Team**: By finding the mode (or using `.value_counts().index[0]`) on the `home_team` column, we can see which team has played the most fixtures at home.
</VSCode.Cell>

<VSCode.Cell language="python">
# Total matches
total_matches = len(df)
print(f"Total matches in dataset: {total_matches}")

# Earliest and Latest Year
df['date'] = pd.to_datetime(df['date'])
earliest_year = df['date'].dt.year.min()
latest_year = df['date'].dt.year.max()
print(f"Earliest year: {earliest_year}")
print(f"Latest year: {latest_year}")

# Unique countries
unique_countries = df['country'].nunique()
print(f"Unique countries: {unique_countries}")

# Most frequent home team
most_frequent_home = df['home_team'].value_counts().index[0]
print(f"Most frequent home team: {most_frequent_home}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
# Goals Analysis
Next, we focus on goal-scoring patterns.

- **Total Goals**: We create a new column `total_goals` by adding `home_score` and `away_score`.
- **Average Goals**: We take the mean of this new column to get the average goals per match.
- **Highest Scoring Match**: We use the maximum value of `total_goals` to filter the DataFrame and extract the row details of the highest scoring fixture.
- **Home vs Away Breakdown**: We sum all `home_score` and `away_score` columns respectively, comparing them to see if home or away teams score more.
- **Mode of Total Goals**: Using `.mode()`, we check the most common number of total goals scored in a match.
</VSCode.Cell>

<VSCode.Cell language="python">
# Create total_goals column
df['total_goals'] = df['home_score'] + df['away_score']

# Average goals per match
average_goals = df['total_goals'].mean()
print(f"Average goals per match: {average_goals:.2f}")

# Highest scoring match details
highest_scoring_match = df.loc[df['total_goals'].idxmax()]
print("\nHighest Scoring Match Details:")
print(highest_scoring_match)

# Total home vs away goals
total_home_goals = df['home_score'].sum()
total_away_goals = df['away_score'].sum()

print(f"\nTotal Home Goals: {total_home_goals}")
print(f"Total Away Goals: {total_away_goals}")
if total_home_goals > total_away_goals:
    print("More goals are scored by the Home team.")
elif total_away_goals > total_home_goals:
    print("More goals are scored by the Away team.")
else:
    print("Home and Away goals are exactly equal.")

# Mode of total goals
mode_goals = df['total_goals'].mode()[0]
print(f"\nMost common total goals (mode): {mode_goals}")
</VSCode.Cell>

<VSCode.Cell language="markdown">
# Match Results
Now we categorize the match outcomes into "Home Win", "Away Win", or "Draw".

- **Result Function**: We define `match_result(row)` and apply it across the DataFrame (setting `axis=1`) to yield a new `result` column.
- **Home Win Percentage**: We count the occurrences of "Home Win" and divide by the total matches to yield a percentage.
- **Home Advantage**: By comparing the count of Home Wins vs Away Wins, we can state whether playing at home gives a historical advantage.
- **Most Wins Historically**: We aggregate wins. A team gets a win when they are the home team and the result is a "Home Win", OR when they are the away team and the result is an "Away Win". We'll combine this data to find the overall most successful team.
</VSCode.Cell>

<VSCode.Cell language="python">
# Create match_result function
def match_result(row):
    if row['home_score'] > row['away_score']:
        return "Home Win"
    elif row['home_score'] < row['away_score']:
        return "Away Win"
    else:
        return "Draw"

# Apply function to create 'result' column
df['result'] = df.apply(match_result, axis=1)

# Percentage of home wins
result_counts = df['result'].value_counts()
home_wins_percent = (result_counts.get("Home Win", 0) / len(df)) * 100
print(f"Percentage of home wins: {home_wins_percent:.2f}%")

# Home advantage comparison
home_wins = result_counts.get("Home Win", 0)
away_wins = result_counts.get("Away Win", 0)
print(f"\nHome wins: {home_wins}, Away wins: {away_wins}")
if home_wins > away_wins:
    print("Statement: A significant home advantage appears to exist historically.")
else:
    print("Statement: There is little to no home advantage historically.")

# Determine which team has the most historical wins
# Filter to only matches with a winner
home_winners = df[df['result'] == 'Home Win']['home_team']
away_winners = df[df['result'] == 'Away Win']['away_team']

# Combine all winning teams
all_winners = pd.concat([home_winners, away_winners])
most_wins_team = all_winners.value_counts().index[0]
most_wins_count = all_winners.value_counts().iloc[0]

print(f"\nTeam with the most historical wins: {most_wins_team} ({most_wins_count} wins)")
</VSCode.Cell>

<VSCode.Cell language="markdown">
# Visualization
Finally, we build visualizations to better understand our dataset using `matplotlib.pyplot`.

- **Histogram of Total Goals**: Shows the distribution of goals scored per match. Setting `bins=15` clumps the data appropriately.
- **Bar Chart of Outcomes**: Displays the occurrences of the three match results.
- **Bar Chart of Top 10 Teams**: We take the top 10 from our aggregated wins list earlier, plot them, and rotate the x-axis labels (`plt.xticks(rotation=45)`) so the team names are fully readable.
</VSCode.Cell>

<VSCode.Cell language="python">
import matplotlib.pyplot as plt

# 1. Histogram of Total Goals
plt.figure(figsize=(10, 5))
plt.hist(df['total_goals'], bins=15, color='skyblue', edgecolor='black')
plt.title('Distribution of Total Goals Per Match')
plt.xlabel('Total Goals')
plt.ylabel('Frequency')
plt.show()

# 2. Bar chart of match outcomes
plt.figure(figsize=(8, 5))
result_counts.plot(kind='bar', color=['#4C72B0', '#55A868', '#C44E52'])
plt.title('Frequency of Match Outcomes')
plt.xlabel('Result')
plt.ylabel('Number of Matches')
plt.xticks(rotation=0)
plt.show()

# 3. Bar chart of the Top 10 teams by total wins
top_10_winners = all_winners.value_counts().head(10)

plt.figure(figsize=(12, 6))
top_10_winners.plot(kind='bar', color='coral')
plt.title('Top 10 Teams by Historical Wins')
plt.xlabel('Team / Country')
plt.ylabel('Total Wins')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()  # Ensure labels do not get cut off
plt.show()
